You can download the `requirements.txt` for this course from the workspace of this lab. `File --> Open...`

# L2: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [3]:
import os
# from utils import get_openai_api_key  # This line is present in the original notebook, however it is not used for local execution

openai_api_key = os.getenv('OPENAI_API_KEY')
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [4]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [5]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [6]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [7]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [8]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [9]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution. 

In [10]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [11]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer:

Title: Unveiling the Latest Trends and Key Players in Artificial Intelligence

Introduction:
- Definition of Artificial Intelligence (AI) and its significance in today's technological landscape
- Brief overview of the rapid advancements and innovations in AI

Key Points:
1. Latest Trends in Artificial Intelligence:
- Machine Learning and Deep Learning: Advancements in algorithms and models
- Natural Language Processing (NLP): Applications in chatbots and language translation
- Computer Vision: Progress in ima

- Display the results of your execution as markdown in the notebook.

In [12]:
from IPython.display import Markdown
Markdown(result)

# Unveiling the Latest Trends and Key Players in Artificial Intelligence

## Introduction: 
Artificial Intelligence (AI) has become a central pillar in today's technological landscape, revolutionizing industries and reshaping the way we interact with technology. Defined as the simulation of human intelligence processes by machines, AI has seen rapid advancements and innovations in recent years, propelling it to the forefront of technological progress. From enhancing decision-making processes to automating mundane tasks, AI continues to redefine the possibilities of what machines can achieve.

## Latest Trends in Artificial Intelligence:

### Machine Learning and Deep Learning:
One of the most significant trends in AI is the evolution of machine learning and deep learning algorithms. These advancements have enabled machines to learn from data, recognize patterns, and make decisions without explicit programming. From self-driving cars to personalized recommendation systems, machine learning and deep learning have revolutionized various industries, driving efficiency and innovation.

### Natural Language Processing (NLP):
Another prominent trend in AI is the application of natural language processing (NLP) in chatbots and language translation services. NLP enables machines to understand, interpret, and generate human language, facilitating seamless communication between humans and machines. With the rise of virtual assistants and language translation tools, NLP has become an essential component of AI applications in everyday life.

### Computer Vision:
Advancements in computer vision have also been a key driver of AI innovation, particularly in image recognition and object detection. By enabling machines to analyze and interpret visual information, computer vision has paved the way for applications such as facial recognition, autonomous vehicles, and medical image analysis. The progress in computer vision capabilities continues to expand the possibilities of AI in diverse fields.

### AI Ethics:
As AI technologies become more pervasive in society, discussions on responsible AI development and use have gained prominence. Ethical considerations surrounding data privacy, bias in algorithms, and the impact of AI on societal norms have sparked debates on the ethical implications of AI. Ensuring that AI systems are developed and deployed ethically is crucial to building trust and fostering positive outcomes for individuals and communities.

## Key Players in the AI Industry:

### Google:
Google stands out as a leader in AI research and development, leveraging its vast resources and expertise to drive innovation in AI technologies. From enhancing search algorithms to developing autonomous systems, Google's contributions to the AI landscape have been instrumental in shaping the industry's trajectory.

### Microsoft:
Microsoft has made significant investments in AI technologies and applications, integrating AI capabilities into its products and services to enhance user experiences and drive business growth. With a focus on democratizing AI and making it accessible to organizations of all sizes, Microsoft continues to be a key player in advancing AI innovation.

### IBM:
IBM is renowned for its Watson AI platform and enterprise solutions, offering a range of AI-powered services to businesses across various sectors. With a strong emphasis on AI ethics and responsible AI practices, IBM has positioned itself as a trusted partner for organizations seeking to harness the power of AI in their operations.

### Amazon:
Amazon has been at the forefront of leveraging AI in e-commerce, cloud services, and voice assistant technology through its Alexa platform. By integrating AI capabilities into its products and services, Amazon has transformed the way customers shop online, interact with technology, and access information, solidifying its position as a key player in the AI industry.

## Noteworthy News in Artificial Intelligence:
Stay tuned for updates on recent breakthroughs in AI research, the implementation of AI in various industries, and the evolving impact of AI on job markets and workforce dynamics. As AI continues to shape the future of technology and society, staying informed about the latest developments and trends is essential for navigating the ever-changing landscape of artificial intelligence.

## Conclusion and Call to Action:
In conclusion, Artificial Intelligence remains a dynamic and transformative force in today's digital age, with endless possibilities for innovation and progress. By staying informed about the latest trends, key players, and ethical considerations in the AI industry, individuals and organizations can harness the power of AI to drive positive change and create a more inclusive and sustainable future. Take action today to explore the vast potential of Artificial Intelligence and contribute to shaping a world powered by intelligent machines.

By following this comprehensive content plan and incorporating SEO keywords naturally throughout the blog post, I aim to provide valuable insights and engaging content for tech enthusiasts, business professionals, students, and anyone interested in the fascinating world of Artificial Intelligence. Let's continue to explore the frontiers of AI together and embrace the opportunities it presents for a brighter future.

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [16]:
topic = "Barbeques in Australia"
result = crew.kickoff(inputs={"topic": topic})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Barbeques in Australia.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer: 

Content Plan Document:

Title: The Ultimate Guide to Barbeques in Australia

Introduction:
- Brief overview of the popularity of barbeques in Australia
- Mention of key players in the barbeque industry in Australia
- Preview of the latest trends in barbeques

Key Points:
1. Latest Trends in Barbeques in Australia:
   - Rise of sustainable and eco-friendly barbeque options
   - Increase in popularity of smoking and slow-cooking techniques
   - Emphasis on outdoor entertainment and cooking experiences

2. Key P

In [17]:
Markdown(result)

# The Ultimate Guide to Barbeques in Australia

## Introduction
Barbeques hold a special place in the hearts of Australians, with outdoor cooking and entertaining being deeply ingrained in the culture. The popularity of barbeques in Australia is undeniable, as families and friends gather around the grill to enjoy delicious food and good company. Key players in the barbeque industry in Australia, such as Weber, Ziegler & Brown, and Everdure by Heston Blumenthal, have been instrumental in shaping the market with their innovative products and offerings. In this ultimate guide, we will explore the latest trends in barbeques, highlight the key players in the industry, and provide noteworthy news on barbeques in Australia.

## Latest Trends in Barbeques in Australia
One of the prominent trends in the barbeque scene in Australia is the rise of sustainable and eco-friendly options. With increasing awareness of environmental issues, many Australians are opting for barbeques that are more environmentally conscious, such as using renewable energy sources or sustainable materials. Another trend gaining traction is the popularity of smoking and slow-cooking techniques, allowing for more flavor depth and complexity in grilled dishes. Additionally, there is an emphasis on outdoor entertainment and cooking experiences, with many Australians investing in outdoor kitchens and dining spaces to enhance their barbeque gatherings.

## Key Players in the Barbeque Industry in Australia
Weber, Ziegler & Brown, and Everdure by Heston Blumenthal are among the top players in the barbeque industry in Australia, known for their quality products and innovative designs. Weber, in particular, has established itself as a household name with its range of gas, charcoal, and electric grills that cater to different preferences and cooking styles. Ziegler & Brown is renowned for its compact and powerful barbeques, perfect for small outdoor spaces or camping trips. Everdure by Heston Blumenthal combines cutting-edge technology with sleek aesthetics to create a unique barbeque experience for enthusiasts.

## Noteworthy News on Barbeques in Australia
Stay updated on the latest barbeque regulations or safety guidelines to ensure a safe and enjoyable cooking experience. Keep an eye out for barbeque events or festivals happening in Australia, where you can discover new recipes, techniques, and products. Look out for collaborations between Australian chefs and barbeque brands, as they often lead to exciting culinary innovations and unique cooking experiences.

## Conclusion
In conclusion, barbeques in Australia are more than just a cooking method – they represent a way of life. By embracing the latest trends, supporting key players in the industry, and staying informed on noteworthy news, you can elevate your barbeque game and create unforgettable memories with loved ones. So fire up the grill, gather your friends and family, and enjoy the ultimate barbeque experience in the land down under.

Remember to explore different barbeque options available in Australia, follow outdoor cooking tips, and share your own barbeque experiences and tips with fellow enthusiasts. Let's keep the flame alive and savor the joy of outdoor cooking together!

*(Opinions expressed in this article are based on the information provided in the content plan and may vary based on individual preferences and experiences.)*

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).